In [1]:
%ls

ibm_ensemble/  ibm_large/  script.ipynb


In [2]:
import os
import numpy as np

In [3]:
# zh-en: total 1976 segments, he-en: 1910 segments
def read_seg_scores(parser="large"): 
    all_scores = {}  
    for filename in os.listdir(f"ibm_{parser}"): 
        assert os.path.exists(f"ibm_{parser}/{filename}")
        with open(f"ibm_{parser}/{filename}", "r", encoding="utf-8") as f:
            lang = filename.split(".")[0]
            sys = filename.split(".")[1]
            print(f"~~~~~ reading file: {filename} ~~~~~")
            print(f"sys: {sys}, lang: {lang}")
            seg_scores = []
            seg_dict = {}
            seg_num = 0
            for line in f.readlines():
                line_l = line.split()
                #print(line_l)
                if "F1:" in line_l: 
                #and ((lang == "zh-en" and seg_num <= 1976) or (lang == "he-en" and seg_num < 1910)):
                    seg_dict[line_l[0]] = float(line_l[line_l.index("F1:")+1])
                    if line.startswith("REENTRANCIES"):
                        seg_dict["avg"] = sum(list(seg_dict.values()))/len(list(seg_dict.values()))
                        seg_scores.append(seg_dict)
                        seg_num += 1
                        seg_dict = {}
                        
            all_scores[(lang, sys)] = seg_scores
    return all_scores
scores_large = read_seg_scores()       
                   

~~~~~ reading file: zh-en.NLLB_MBR_BLEU.smatch++.txt ~~~~~
sys: NLLB_MBR_BLEU, lang: zh-en
~~~~~ reading file: he-en.NLLB_MBR_BLEU.smatch++.txt ~~~~~
sys: NLLB_MBR_BLEU, lang: he-en
~~~~~ reading file: zh-en.ZengHuiMT.smatch++.txt ~~~~~
sys: ZengHuiMT, lang: zh-en
~~~~~ reading file: zh-en.ANVITA.smatch++.txt ~~~~~
sys: ANVITA, lang: zh-en
~~~~~ reading file: zh-en.ONLINE-B.smatch++.txt ~~~~~
sys: ONLINE-B, lang: zh-en
~~~~~ reading file: he-en.ONLINE-B.smatch++.txt ~~~~~
sys: ONLINE-B, lang: he-en
~~~~~ reading file: zh-en.ONLINE-M.smatch++.txt ~~~~~
sys: ONLINE-M, lang: zh-en
~~~~~ reading file: zh-en.ONLINE-G.smatch++.txt ~~~~~
sys: ONLINE-G, lang: zh-en
~~~~~ reading file: he-en.GPT4-5shot.smatch++.txt ~~~~~
sys: GPT4-5shot, lang: he-en
~~~~~ reading file: he-en.ZengHuiMT.smatch++.txt ~~~~~
sys: ZengHuiMT, lang: he-en
~~~~~ reading file: zh-en.synthetic_ref.smatch++.txt ~~~~~
sys: synthetic_ref, lang: zh-en
~~~~~ reading file: zh-en.NLLB_Greedy.smatch++.txt ~~~~~
sys: NLLB_Greedy, 

In [4]:
scores_ensemble = read_seg_scores("ensemble")
len(list(scores_large.values())[0])

~~~~~ reading file: he-en.ONLINE-B.smatch++.ensemble.txt ~~~~~
sys: ONLINE-B, lang: he-en
~~~~~ reading file: zh-en.IOL_Research.smatch++.ensemble.txt ~~~~~
sys: IOL_Research, lang: zh-en
~~~~~ reading file: zh-en.ONLINE-W.smatch++.ensemble.txt ~~~~~
sys: ONLINE-W, lang: zh-en
~~~~~ reading file: he-en.UvA-LTL.smatch++.ensemble.txt ~~~~~
sys: UvA-LTL, lang: he-en
~~~~~ reading file: zh-en.Lan-BridgeMT.smatch++.ensemble.txt ~~~~~
sys: Lan-BridgeMT, lang: zh-en
~~~~~ reading file: he-en.refB.smatch++.ensemble.txt ~~~~~
sys: refB, lang: he-en
~~~~~ reading file: zh-en.GPT4-5shot.smatch++.ensemble.txt ~~~~~
sys: GPT4-5shot, lang: zh-en
~~~~~ reading file: he-en.ONLINE-Y.smatch++.ensemble.txt ~~~~~
sys: ONLINE-Y, lang: he-en
~~~~~ reading file: zh-en.synthetic_ref.smatch++.ensemble.txt ~~~~~
sys: synthetic_ref, lang: zh-en
~~~~~ reading file: zh-en.HW-TSC.smatch++.ensemble.txt ~~~~~
sys: HW-TSC, lang: zh-en
~~~~~ reading file: zh-en.ONLINE-A.smatch++.ensemble.txt ~~~~~
sys: ONLINE-A, lang: 

1978

In [8]:
scores_large[("he-en", "GPT4-5shot")][-2]

{'AGENT': 65.53,
 'CAUSE': 56.1,
 'CONCEPT': 72.82,
 'FOCUS': 62.5,
 'INSTRUMENT': 21.84,
 'LOCATION': 63.02,
 'MATHS': 28.57,
 'NER': 76.11,
 'PATIENT': 66.15,
 'POLARITY': 67.64,
 'POSSESSION': 45.79,
 'PURPOSE': 42.9,
 'QUANTIFIER': 68.35,
 'QUESTION': 67.31,
 'SRL-core': 68.44,
 'TIME': 63.69,
 'TOPIC': 39.56,
 'WIKI': 100.0,
 'REENTRANCIES': 62.38,
 'avg': 59.93157894736842}

In [15]:
def to_file(scores=scores_large, parser="large"):    
    micro_scores, seg_scores = {}, {}
    for lang, sys in scores:
        for seg_score_dict in list(scores[(lang, sys)])[:-2]:
            # the last two score_dict is for micro and macro system scores
            if lang not in seg_scores:
                seg_scores[lang] = []
            seg_scores[lang].append((sys, seg_score_dict["avg"]/100))
            
        micro_score_dict = list(scores[(lang, sys)])[-2]
        if lang not in micro_scores:
            micro_scores[lang] = []
        micro_scores[lang].append((sys, micro_score_dict["avg"]/100))

    for lang in micro_scores:
        with open(f"{lang}.{parser}.sys.score", "w", encoding="utf-8") as f:
            for sys, sys_score in micro_scores[lang]:
                f.write(sys)
                f.write("\t")
                f.write(str(sys_score))
                f.write("\n")
        with open(f"{lang}.{parser}.seg.score", "w", encoding="utf-8") as f2:
            for sys, seg_score in seg_scores[lang]:
                f2.write(sys)
                f2.write("\t")
                f2.write(str(seg_score))
                f2.write("\n")
               

In [16]:
to_file(scores_large, "large")
to_file(scores_ensemble, "ensemble")